# Train the deployed pooled ridge model

The production model behind `em_agent`'s surprise graph. It is the **pooled** ridge from `thin_preview.ipynb`: every event appears twice, once with features from the competition's preview (`src_own = 0`) and once from a thin preview (`src_own = 1`), so it serves both the competition's previews and our own.

- **Data:** all 1,364 events (train and test); evaluation is finished, so the deployed model uses everything.
- **Settings:** the configuration cross-validation chose on train in `thin_preview.ipynb` (feature set `all`, alpha 3,162, raw target, no log transform), so test is not used for tuning.
- **Output:** `src/em_agent/artifacts/pooled_ridge_v2.joblib` (+ `.json` description), loaded by the agent with `ml.model.load_model`, and a frozen copy of the v2 taxonomy, so the deployed app never reads `data/`.

In [ ]:
import shutil
from pathlib import Path

import pandas as pd

from ml import model, preprocess
from suprise_dim import data, dataset, extraction, features, grouping

ARTIFACTS = data.ROOT / "src" / "em_agent" / "artifacts"
MODEL_PATH = ARTIFACTS / "pooled_ridge_v2.joblib"
CONFIG = {"feature_set": "all", "alpha": 3162.27766, "target_mode": "raw", "log_counts": False}
THIN = "thin_v1"

split = data.load_split()
events_by_id = {e["event_id"]: e for e in data.load_eligible_events()}
tax = grouping.load_taxonomy("v2")
labels = {e: "train" for e in split["train"]} | {e: "test" for e in split["test"]}

In [2]:
# Events whose thin extraction is complete; both versions of each.
assigned = set(grouping.load_assignments(f"v2_{THIN}")["item_id"])
records = [r for r in extraction.load_extractions(data.extractions_path(THIN)) if r["event_id"] in labels]
complete = {r["event_id"] for r in records if all(f"{r['event_id']}#{i}" in assigned for i in range(len(r["surprises"])))}

surprises_thin = dataset.build_extraction_frame(THIN, f"v2_{THIN}", {e: labels[e] for e in complete})
split_like = {"train": [e for e in split["train"] if e in complete], "test": [e for e in split["test"] if e in complete]}
feat_thin = features.build_event_features(surprises_thin, split_like, events_by_id, tax)
feat_full = pd.read_csv(features.features_path("v2"))
feat_full = feat_full[feat_full["event_id"].isin(complete)]

pooled = pd.concat([preprocess.add_source_features(feat_full.reset_index(drop=True), "competition"),
                    preprocess.add_source_features(feat_thin.reset_index(drop=True), "own")], ignore_index=True)
feature_list = preprocess.with_source(preprocess.feature_sets(feat_full.columns)[CONFIG["feature_set"]], pooled.columns)
print(f"{len(complete)} events x 2 sources = {len(pooled)} rows | {len(feature_list)} features")

1364 events x 2 sources = 2728 rows | 64 features


In [ ]:
fitted = model.fit(pooled, feature_list, CONFIG["alpha"], target_mode=CONFIG["target_mode"], log_counts=CONFIG["log_counts"])
model.save_model(fitted, MODEL_PATH, **CONFIG, taxonomy_version="v2", thin_previews=THIN,
                 n_events=len(complete), n_rows=len(pooled), trained_on="all 2026Q3 preview events (train + test)")
shutil.copy(grouping.taxonomy_path("v2"), ARTIFACTS / "taxonomy_v2.json")

reloaded = model.load_model(MODEL_PATH)
same = (reloaded.predict(pooled) == fitted.predict(pooled)).all()
print(f"saved {MODEL_PATH.relative_to(data.ROOT)} | reload gives identical predictions: {same}")
fitted.coefficients().head(10).round(4).to_frame("coef")